# Experiment H16: LR Confound Audit -- Are Spectral Democracy Results Robust to LR Matching?

---

## Theoretical Background

### The LR Confound Problem

A critical lesson from Experiment H6 was that the reported "130x curvature rescaling"
difference between Muon and SGD was actually an **LR artifact**: when both optimizers
were trained at their respective optimal LRs, the effect largely disappeared. This
raised a methodological concern: **are ANY of the spectral comparisons between Muon
and SGD contaminated by suboptimal LR choices?**

This experiment applies the same scrutiny to two key spectral metrics from the 1.3
experiment series:

1. **Effective rank** (spectral democracy): $\text{erank}(W) = \exp\left(-\sum_i p_i \log p_i\right)$
   where $p_i = \sigma_i / \sum_j \sigma_j$. Higher effective rank means more
   democratic (equalized) singular value distribution. Muon was reported to produce
   higher effective rank than SGD.

2. **Condition number** $\kappa(W) = \sigma_1 / \sigma_d$. Muon was reported to produce
   lower condition numbers (better-conditioned matrices).

### The Falsification Logic

This is a **Tier 4 falsification stress test**. If the spectral differences between
Muon and SGD change qualitatively (flip sign or change by > 50%) when both use their
optimal LR, then the original comparisons were confounded and the spectral democracy
narrative needs revision.

If the differences are robust to LR matching, this strengthens confidence that they
reflect genuine mechanistic differences rather than optimization quality artifacts.

## Experimental Design

| Configuration | Optimizer | LR | Purpose |
|:--------------|:----------|:---|:--------|
| `sgd_default` | SGD | 0.01 (default) | Original comparison baseline |
| `sgd_optimal` | SGD | Best from sweep | LR-matched baseline |
| `muon_default` | Muon | 0.02 (default) | Original comparison |
| `muon_optimal` | Muon | Best from sweep | LR-matched Muon |

Phase 1 sweeps LR for each optimizer. Phase 2 trains all four configurations on
10 seeds and measures spectral metrics per layer.

## Key Tests

| Test | Question | Falsification criterion |
|:-----|:---------|:----------------------|
| T1 | Effective rank difference robust? | Difference changes > 50% with LR matching |
| T2 | Kappa ranking flips? | Muon < SGD at default but Muon > SGD at optimal (or vice versa) |

**T1 PASS = confound exists** (spectral democracy narrative weakened).
**T2 PASS = ranking flip** (spectral comparison completely invalidated).

## Environment Setup

Import NumPy for linear algebra. This experiment uses 10 seeds (more than the
standard 5) for extra statistical power, since we are looking for subtle changes
in spectral metrics that could be masked by noise.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

**Network**: 4-layer deep linear, 32x32, 500 steps with momentum 0.9.

**LR sweep**: 10 candidate LRs per optimizer, spanning different ranges because
SGD and Muon have different effective step sizes. The first 3 seeds are used for
LR selection; all 10 seeds for final evaluation.

**Default LRs**: The "default" values (SGD: 0.01, Muon: 0.02) represent the LRs
used in earlier experiments. If optimal LRs differ significantly from defaults,
any spectral metric differences at default LR could be artifacts.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 10
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_LAYERS = {NUM_LAYERS}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  MOMENTUM = {MOMENTUM}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  NS_ITERS = {NS_ITERS}")


In [ ]:
MUON_LRS = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]
SGD_LRS = [0.2, 0.1, 0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.001, 0.0005]

In [ ]:
DEFAULT_MUON_LR = 0.02
DEFAULT_SGD_LR = 0.01

## Core Algorithms

### Newton-Schulz Orthogonalization
Standard NS iteration for Muon's gradient orthogonalization.

### Weight Initialization
Near-identity initialization $W = I + 0.1\epsilon$ for all layers.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

## Network Architecture

Forward pass, MSE loss, and backpropagation for a 4-layer deep linear network.
Standard chain $\hat{Y} = W_4 W_3 W_2 W_1 X$.

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

## Spectral Metrics

### Effective Rank (Spectral Democracy)

The effective rank measures how "democratic" the singular value distribution is:
$$\text{erank}(W) = \exp\left(H\right), \quad H = -\sum_i p_i \log p_i, \quad p_i = \frac{\sigma_i}{\sum_j \sigma_j}$$

- Maximum effective rank = $d$ (all SVs equal, perfectly democratic)
- Minimum effective rank = 1 (only one non-zero SV, rank-1)
- Muon is predicted to produce higher effective rank (SV equalization)

### Condition Number

$\kappa(W) = \sigma_1 / \sigma_d$. Lower is better-conditioned. Muon's orthogonal
updates should produce $\kappa \approx 1$ (all SVs near 1), while SGD's anisotropic
updates may produce $\kappa \gg 1$.

The question is: are these differences genuine mechanistic effects, or do they
change when both optimizers use their best possible LR?

In [ ]:
def effective_rank(W):
    """Effective rank = exp(entropy of normalized SVs)."""
    s = np.linalg.svd(W, compute_uv=False)
    s = s[s > 1e-12]
    if len(s) == 0:
        return 0
    p = s / np.sum(s)
    H = -np.sum(p * np.log(p + 1e-30))
    return np.exp(H)

In [ ]:
def condition_number(W):
    s = np.linalg.svd(W, compute_uv=False)
    return s[0] / max(s[-1], 1e-12)

## Training Loop and Data Generation

The training loop runs for 500 steps with momentum, supporting both SGD and Muon.
Returns both the final weights (for spectral analysis) and the final loss (for
LR selection). A `None` return for weights indicates divergence.

In [ ]:
def train(weights_init, X, Y, lr, optimizer):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return None, float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(len(weights)):
            if optimizer == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return weights, compute_loss(weights, X, Y)

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = rng.randn(DIM, BATCH_SIZE) * 0.3
    return X, Y

## Main Experiment: Default vs Optimal LR Comparison

The experiment proceeds in two phases:

**Phase 1 -- LR Sweep**: Find the optimal LR for each optimizer using the first 3
seeds. This identifies whether the default LRs are actually near-optimal or significantly
suboptimal.

**Phase 2 -- Full Training**: Train all four configurations (SGD/Muon x default/optimal LR)
on all 10 seeds. Measure effective rank and condition number per layer.

The critical comparison is between the **default-LR** spectral metrics (which were used
in earlier experiments) and the **optimal-LR** spectral metrics. If the differences
change qualitatively, the earlier conclusions were confounded.

### What "confounded" means precisely

If at default LR, SGD was undertrained (higher loss, worse spectral metrics) while
Muon was well-trained, then Muon's spectral superiority could be an artifact of
better optimization quality, not a fundamental property of the NS orthogonalization.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H16: LR CONFOUND AUDIT -- Spectral Democracy and Condition Number")
    print("=" * 100)
    print(f"Network: {NUM_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"Default LRs: SGD={DEFAULT_SGD_LR}, Muon={DEFAULT_MUON_LR}")
    print()

    # LR sweep
    print("Phase 1: LR sweep (using first 3 seeds)...")
    best_lrs = {}
    for opt, candidates in [('muon', MUON_LRS), ('sgd', SGD_LRS)]:
        best_lr, best_loss = candidates[-1], float('inf')
        all_sweeps = []
        for lr in candidates:
            losses = []
            for s in seeds[:3]:
                X, Y = make_data(s)
                w = init_weights(s + 5000)
                _, fl = train(w, X, Y, lr, opt)
                losses.append(fl)
            ml = np.mean([l for l in losses if np.isfinite(l)]) if any(np.isfinite(l) for l in losses) else float('inf')
            all_sweeps.append((lr, ml))
            if ml < best_loss:
                best_loss = ml
                best_lr = lr
        best_lrs[opt] = best_lr
        print(f"  {opt}: best_lr={best_lr:.4f} (loss={best_loss:.4e})")
        # Show how far default was from optimal
        default_lr = DEFAULT_MUON_LR if opt == 'muon' else DEFAULT_SGD_LR
        default_loss = [loss for lr, loss in all_sweeps if abs(lr - default_lr) < 1e-6]
        if default_loss:
            print(f"    Default LR={default_lr} loss={default_loss[0]:.4e}  (ratio to optimal: {default_loss[0]/max(best_loss,1e-30):.2f}x)")
        if best_lr != default_lr:
            print(f"    NOTE: Optimal LR differs from default! ({best_lr} vs {default_lr})")
        else:
            print(f"    Default LR IS optimal.")

    # Full training at both default and optimal LR
    print(f"\nPhase 2: Full training (all {NUM_SEEDS} seeds, 4 configurations)...")

    configs = [
        ('sgd_default', 'sgd', DEFAULT_SGD_LR),
        ('sgd_optimal', 'sgd', best_lrs['sgd']),
        ('muon_default', 'muon', DEFAULT_MUON_LR),
        ('muon_optimal', 'muon', best_lrs['muon']),
    ]

    results = {}
    for name, opt, lr in configs:
        all_erank = {l: [] for l in range(NUM_LAYERS)}
        all_kappa = {l: [] for l in range(NUM_LAYERS)}
        all_loss = []

        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            final_w, fl = train(w, X, Y, lr, opt)
            all_loss.append(fl)
            if final_w is not None:
                for l in range(NUM_LAYERS):
                    all_erank[l].append(effective_rank(final_w[l]))
                    all_kappa[l].append(condition_number(final_w[l]))

        results[name] = {
            'loss': np.mean(all_loss),
            'erank': {l: np.mean(all_erank[l]) for l in range(NUM_LAYERS)},
            'kappa': {l: np.mean(all_kappa[l]) for l in range(NUM_LAYERS)},
            'lr': lr,
        }
        avg_erank = np.mean([results[name]['erank'][l] for l in range(NUM_LAYERS)])
        avg_kappa = np.mean([results[name]['kappa'][l] for l in range(NUM_LAYERS)])
        print(f"  {name:>16}: loss={results[name]['loss']:.4e}, avg_erank={avg_erank:.2f}, avg_kappa={avg_kappa:.2f} (lr={lr})")

    # Results
    print(f"\n{'=' * 100}")
    print("SPECTRAL METRICS AT DEFAULT vs OPTIMAL LR")
    print(f"{'=' * 100}")

    for metric_name, metric_key in [('Effective Rank', 'erank'), ('Condition Number', 'kappa')]:
        print(f"\n  {metric_name}:")
        print(f"  {'Config':>20}  {'LR':>8}", end='')
        for l in range(NUM_LAYERS):
            print(f"  {'L'+str(l):>8}", end='')
        print(f"  {'Mean':>8}  {'Loss':>12}")
        print("  " + "-" * (30 + 10 * NUM_LAYERS + 22))

        for name, _, _ in configs:
            r = results[name]
            avg = np.mean([r[metric_key][l] for l in range(NUM_LAYERS)])
            print(f"  {name:>20}  {r['lr']:>8.4f}", end='')
            for l in range(NUM_LAYERS):
                print(f"  {r[metric_key][l]:>8.2f}", end='')
            print(f"  {avg:>8.2f}  {r['loss']:>12.4e}")

    # Detailed difference analysis
    print(f"\n  Difference analysis (Muon - SGD):")
    for label, muon_key, sgd_key in [('Default LR', 'muon_default', 'sgd_default'),
                                      ('Optimal LR', 'muon_optimal', 'sgd_optimal')]:
        erank_diffs = [results[muon_key]['erank'][l] - results[sgd_key]['erank'][l] for l in range(NUM_LAYERS)]
        kappa_ratios = [results[muon_key]['kappa'][l] / max(results[sgd_key]['kappa'][l], 1e-12) for l in range(NUM_LAYERS)]
        print(f"    {label}:")
        print(f"      Erank diff (Muon-SGD): {['%.2f' % d for d in erank_diffs]}  mean={np.mean(erank_diffs):.3f}")
        print(f"      Kappa ratio (Muon/SGD): {['%.2f' % r for r in kappa_ratios]}  mean={np.mean(kappa_ratios):.3f}")

    # Key comparison: does the ranking change?
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    # Effective rank difference at default vs optimal
    erank_diff_default = np.mean([results['muon_default']['erank'][l] - results['sgd_default']['erank'][l] for l in range(NUM_LAYERS)])
    erank_diff_optimal = np.mean([results['muon_optimal']['erank'][l] - results['sgd_optimal']['erank'][l] for l in range(NUM_LAYERS)])

    t1 = abs(erank_diff_default - erank_diff_optimal) > abs(erank_diff_default) * 0.5
    print(f"\n  T1: Effective rank difference changes >50% with LR matching?")
    print(f"      Default diff (Muon-SGD): {erank_diff_default:.3f}")
    print(f"      Optimal diff (Muon-SGD): {erank_diff_optimal:.3f}")
    print(f"      Change: {abs(erank_diff_default - erank_diff_optimal):.3f} ({abs(erank_diff_default - erank_diff_optimal)/max(abs(erank_diff_default), 1e-12)*100:.0f}% of default)")
    print(f"      --> {'PASS (LR confound detected -- spectral democracy partially an artifact)' if t1 else 'FAIL (robust -- spectral democracy is a genuine Muon property)'}")
    if not t1:
        print(f"      Interpretation: The effective rank advantage of Muon over SGD is ROBUST to")
        print(f"      LR matching. This means spectral democracy is a genuine consequence of the")
        print(f"      Newton-Schulz orthogonalization, not an artifact of optimization quality.")
    else:
        print(f"      Interpretation: The effective rank difference CHANGES substantially when both")
        print(f"      optimizers use their optimal LR. The original comparison may have been comparing")
        print(f"      'well-optimized Muon' vs 'poorly-optimized SGD'.")

    # Kappa ranking flip?
    kappa_ratio_default = np.mean([results['muon_default']['kappa'][l] / max(results['sgd_default']['kappa'][l], 1e-12) for l in range(NUM_LAYERS)])
    kappa_ratio_optimal = np.mean([results['muon_optimal']['kappa'][l] / max(results['sgd_optimal']['kappa'][l], 1e-12) for l in range(NUM_LAYERS)])

    flip = (kappa_ratio_default < 1.0) != (kappa_ratio_optimal < 1.0)
    t2 = flip
    print(f"\n  T2: Kappa ratio flips qualitatively with LR matching?")
    print(f"      Default Muon/SGD kappa ratio: {kappa_ratio_default:.3f} ({'Muon better' if kappa_ratio_default < 1 else 'SGD better'})")
    print(f"      Optimal Muon/SGD kappa ratio: {kappa_ratio_optimal:.3f} ({'Muon better' if kappa_ratio_optimal < 1 else 'SGD better'})")
    print(f"      --> {'PASS (RANKING FLIPS -- conditioning comparison invalidated)' if t2 else 'FAIL (ranking stable -- conditioning comparison robust)'}")
    if not t2:
        print(f"      Interpretation: The condition number comparison is STABLE across LR choices.")
        print(f"      Muon's conditioning advantage (or disadvantage) is a genuine property,")
        print(f"      not an artifact of LR mismatch.")
    else:
        print(f"      Interpretation: CRITICAL FINDING. The condition number ranking REVERSES when")
        print(f"      both optimizers use optimal LR. The original comparison was fundamentally")
        print(f"      confounded by LR choice.")

    # Summary
    print(f"\n\n{'=' * 100}")
    print("SUMMARY")
    print(f"{'=' * 100}")
    confounds_found = sum([t1, t2])
    if confounds_found == 0:
        print(f"  No LR confounds detected. Both spectral democracy (effective rank)")
        print(f"  and conditioning (kappa) comparisons are ROBUST to LR matching.")
        print(f"  The spectral properties previously reported are genuine Muon features.")
    elif confounds_found == 1:
        print(f"  PARTIAL confound: one of the two spectral metrics is affected by LR choice.")
        print(f"  {'Effective rank' if t1 else 'Condition number'} comparison is confounded;")
        print(f"  {'condition number' if t1 else 'effective rank'} comparison is robust.")
    else:
        print(f"  BOTH spectral metrics are confounded by LR choice!")
        print(f"  The spectral democracy narrative from earlier experiments may need")
        print(f"  substantial revision.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions

### What This Experiment Tests

This is a **methodological stress test** -- the hardest kind of experiment to run
because it asks "were our earlier results wrong?" Finding a confound is scientifically
valuable (it corrects the record) but uncomfortable.

### Interpretation Guide

- **No confounds found (T1 FAIL, T2 FAIL)**: The strongest possible result for the
  theory. Both spectral democracy and conditioning comparisons survive the LR
  confound audit. These are genuine properties of the Newton-Schulz orthogonalization,
  not artifacts of LR mismatch. The Tier 1-3 results that depend on these metrics
  are on solid ground.

- **T1 PASS (erank confounded)**: The spectral democracy (effective rank) comparison
  is sensitive to LR choice. Conclusions from Exp 1.3a-i and related experiments
  should be re-evaluated with LR-matched comparisons.

- **T2 PASS (kappa flips)**: The condition number comparison reverses with LR matching.
  This is a serious issue: Experiments that relied on Muon producing lower $\kappa$
  than SGD may have been comparing optimization quality, not mechanism.

- **Both confounded**: The spectral narrative needs substantial revision. However,
  note that even if spectral metrics are confounded, Muon's directional advantage
  (from H6b, H21b) may still be genuine -- it is measured differently.

### Connection to Other Experiments

- **H6/H6a**: The original LR confound audit that motivated this experiment
- **Exp 1.3a-i**: Effective rank layer-by-layer (uses these spectral metrics)
- **Exp 1.3c**: WeightWatcher alpha comparisons (related concern)
- **SPECTRAL_DEMOCRACY_test**: Another spectral democracy measurement
- **H15a/b**: Direction quality and SVD clamping (alternative metrics less
  susceptible to LR confounds)

### Broader Lesson

Every comparison between optimizers should be done at **matched optimal LR**.
Default LRs may accidentally favor one optimizer, creating spurious metric
differences that are really optimization quality differences in disguise.